In [ ]:
from urllib.request import urlopen
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware
from deepagents.backends import StateBackend
from langgraph.checkpoint.memory import InMemorySaver


In [ ]:
from pathlib import Path
from typing import Dict, Any
from deepagents.backends.utils import create_file_data

def get_skills_from_path(path: str) -> Dict[str, Any]:
    base_path = Path(path)
    skills = {}
    
    if base_path.exists() and base_path.is_dir():
        for p in base_path.rglob("*"):
            if p.is_file() and p.name.lower() == "skill.md":
                # Extract relative folder names (e.g., 'aws', 'kubernetes')
                skill_folder = p.parent.relative_to(base_path).as_posix()
                
                virtual_path = f"/skills/{skill_folder}/SKILL.md"
                
                skills[virtual_path] = create_file_data(p.read_text("utf-8"))
                
    return skills

# Execution
skill_files = get_skills_from_path("skills")
skill_files

In [15]:
d_agent = create_deep_agent(
    model="groq:Openai/gpt-oss-20b",
    backend = StateBackend(),
    skills=["skills"],
    checkpointer=InMemorySaver()
)
result = d_agent.invoke({
    "messages": [
        {
        "role": "user", 
        "content": "what skills do you have and when do you use each provide each skills you have"
        }
    ],
    "files": skill_files
},
config={"configurable": {"thread_id": "2"}}
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m4jqkfz9erttrhz3rzqn3fp2` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199525, Requested 2374. Please try again in 13m40.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
print(result)
pretty_print_ai_response(result)

In [ ]:
import textwrap

def pretty_print_ai_response(response_input, width=90):
    response_text = ""

    # 1. If it's a full LangGraph dictionary trace
    if isinstance(response_input, dict):
        # Look for messages first
        if "messages" in response_input and response_input["messages"]:
            # Traverse backward to find the first message with actual text content
            for msg in reversed(response_input["messages"]):
                content = getattr(msg, 'content', '') or (msg.get('content', '') if isinstance(msg, dict) else '')
                if content.strip():
                    response_text = content
                    break
        
        # Fallback to the 'files' section if messages are empty tool invocations
        if not response_text and "files" in response_input:
            for file_path, file_data in response_input["files"].items():
                # Extract ONLY the text content string, completely ignoring metadata keys
                if isinstance(file_data, dict) and "content" in file_data:
                    response_text = file_data["content"]
                    print(f"\n--- Displaying File: {file_path} ---")
                    break
                elif isinstance(file_data, str):
                    response_text = file_data
                    print(f"\n--- Displaying File: {file_path} ---")
                    break
    
    # 2. If it's an LangChain/LangGraph Message instance directly
    elif hasattr(response_input, 'content'):
        response_text = response_input.content
        
    # 3. If it's already a raw string
    elif isinstance(response_input, str):
        response_text = response_input
    
    # Fallback if no printable content was resolved
    if not response_text or not response_text.strip():
        print("⚠️ No printable text content found.")
        return

    # --- TABULAR RENDER ENGINE ---
    lines = response_text.splitlines()
    processed_lines = []
    table_buffer = []
    in_table = False

    def flush_table(buffer):
        if not buffer: return []
        cleaned_rows = []
        for row in buffer:
            cells = [cell.strip() for cell in row.split('|')]
            if row.strip().startswith('|'): cells = cells[1:]
            if row.strip().endswith('|'): cells = cells[:-1]
            cleaned_rows.append(cells)
            
        max_cols = max(len(row) for row in cleaned_rows)
        col_widths = [0] * max_cols
        
        for row in cleaned_rows:
            if all(set(cell).issubset({'-', ':', ' '}) for cell in row): continue
            for i, cell in enumerate(row):
                if i < len(col_widths):
                    col_widths[i] = max(col_widths[i], len(cell))
                    
        formatted_table = []
        for row in cleaned_rows:
            if all(set(cell).issubset({'-', ':', ' '}) for cell in row):
                divider_cells = ['-' * (col_widths[i] + 2) if i < len(col_widths) else '-----' for i in range(max_cols)]
                formatted_table.append("  |" + "|".join(divider_cells) + "|")
                continue
            padded_cells = [f" {row[i].ljust(col_widths[i])} " if i < len(row) else f" {''.ljust(col_widths[i])} " for i in range(max_cols)]
            formatted_table.append("  |" + "|".join(padded_cells) + "|")
        return formatted_table

    for line in lines:
        if '|' in line:
            in_table = True
            table_buffer.append(line)
        else:
            if in_table:
                processed_lines.extend(flush_table(table_buffer))
                table_buffer, in_table = [], False
            if not line.strip():
                processed_lines.append('')
            else:
                processed_lines.append(textwrap.fill(line, width=width))
                
    if in_table:
        processed_lines.extend(flush_table(table_buffer))

    print('\n'.join(processed_lines))
